In [4]:
import os
from typing import TypedDict, Annotated, Sequence
from datetime import datetime, timedelta
import json
from openai import OpenAI
from duckduckgo_search import DDGS
from langgraph.graph import Graph, END
from langchain_core.messages import HumanMessage, AIMessage
from langchain_groq import ChatGroq
import requests

In [5]:
GROQ_API_KEY = "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"
TEAMS_WEBHOOK_URL = "https://testmaq.webhook.office.com/webhookb2/77dbbf0d-7656-4c92-bfc1-3ed6c796c5af@e4d98dd2-9199-42e5-ba8b-da3e763ede2e/IncomingWebhook/b5a4f8810ba74ff09b2e5493cbada038/9c3e2c58-f3a5-4a17-b228-ac503504e338/V22ixM-uDStJHdartGsGLFrYnqOey19xB6J3cLTj5CXwg1"
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["TEAMS_WEBHOOK_URL"] = TEAMS_WEBHOOK_URL


In [6]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TEAMS_WEBHOOK_URL = os.getenv("TEAMS_WEBHOOK_URL")


In [7]:
class AgentState(TypedDict):
    topic: str
    search_results: list
    news_summary: str
    messages: Sequence[HumanMessage | AIMessage]

def search_news(state: AgentState) -> list:
    """Search news using DuckDuckGo"""
    with DDGS() as ddgs:
        results = list(ddgs.news(
            keywords=state['topic'],
            max_results=15,
            timelimit='w'
        ))
    state['search_results'] = results
    return state

In [8]:
def create_news_summary(state: AgentState) -> AgentState:
    """Generate news summary using Groq"""
    chat = ChatGroq(groq_api_key=GROQ_API_KEY, model_name='llama3-8b-8192')
    
    # Format the search results to include only relevant information
    formatted_results = [
        {
            'title': result.get('title', ''),
            'body': result.get('body', ''),
            'url': result.get('url', ''),
            'date': result.get('date', '')
        }
        for result in state['search_results']
    ]
    
    prompt = f"""Create a top 10 news highlights from the past week for the topic: {state['topic']}
    Based on these search results: {json.dumps(formatted_results)}
    Format as markdown with and sort for latest update first:
    1. Clear headline for each item
    2. Brief 1-2 sentence summary
    3. Source attribution
    4. Add URL also as a hyperlink
    """
    response = chat.invoke(prompt)
    state["news_summary"] = response.content
    return state

In [9]:
def post_to_teams(state: AgentState) -> AgentState:
    """Post news summary to Teams channel"""
    payload = {
        "text": f"# News Update: {state['topic']}\n\n{state['news_summary']}"
    }
    response = requests.post(
        TEAMS_WEBHOOK_URL,
        json=payload
    )
    response.raise_for_status()
    return state

In [10]:
def create_workflow():
    """Create and configure the workflow graph"""
    workflow = Graph()
    workflow.set_entry_point("search")
    workflow.add_node("search", search_news)
    workflow.add_node("summarize", create_news_summary)
    workflow.add_node("post", post_to_teams)
    workflow.add_edge("search", "summarize")
    workflow.add_edge("summarize", "post")
    workflow.add_edge("post", END)
    return workflow.compile()

In [11]:
def run_agent(topic: str):
    """Execute the news agent workflow"""
    # Initialize state
    state = AgentState(
        topic=topic,
        search_results=[],
        news_summary="",
        messages=[]
    )
    # Create and run workflow
    workflow = create_workflow()
    final_state = workflow.invoke(state)
    return final_state

In [12]:
if __name__ == "__main__":
    topic = input("Enter news topic: ")
    result = run_agent(topic)
    print(f"News summary posted to Teams:\n{result['news_summary']}")


News summary posted to Teams:
Here are the top 10 news highlights from the past week for the topic: Cricket, in markdown format, sorted for latest update first:

### 1. Michael Clarke Handed Ultimate Honour as Cricket World Reacts to News about Aussie Legend

Michael Clarke, former Australian cricket captain, has been inducted into the Australian Cricket Hall of Fame. Clarke, known for his outstanding performances, becomes the 64th inductee to the Hall of Fame. [Source: MSN](https://www.msn.com/en-au/sport/cricket/michael-clarke-handed-ultimate-honour-as-cricket-world-reacts-to-news-about-aussie-legend/ar-AA1xGrZj)

### 2. Abhishek Sharma Smashes 79 Runs as India Beats England by 7 Wickets in First T20

India opener Abhishek Sharma's sensational power-hitting secured India's comprehensive seven-wicket victory against England in the opening T20 International at the Eden Gardens. [Source: MSN](https://www.msn.com/en-us/sports/other/cricket-india-blitz-england-led-by-abhishek-s-superb-hit

In [ ]:
["2024-06-01", "2024-06-08", "2024-06-15", "2024-06-22", "2024-06-29", "2024-07-06", "2024-07-13", "2024-07-20", "2024-07-27", "2024-08-03", "2024-08-10", "2024-08-17", "2024-08-24", "2024-08-31", "2024-09-07", "2024-09-14", "2024-09-21", "2024-09-28", "2024-10-05", "2024-10-12", "2024-10-19", "2024-10-26", "2024-11-02", "2024-11-09", "2024-11-16", "2024-11-23", "2024-11-30"]